# Data Quality Findings & Initial Data Pipeline

Proyek Kelompok 7: Karakteristik Ekspresi Gen Berdasarkan Sampel (Reduksi Dimensi dan Pencarian Pola, fokus obat).

Notebook ini berisi dua bagian:
1. **Data Quality Findings**: enam masalah kualitas data yang kami temukan pada `GDSC_DATASET.csv` dan `CCLE_RNAseq_rsem_genes_tpm_20180929.txt.gz`, lengkap dengan kode pembuktiannya.
2. **Initial Data Pipeline**: alur awal dari data mentah sampai matriks ekspresi yang siap dipakai untuk reduksi dimensi.

# Bagian 1. Data Quality Findings

Tiap temuan dibuktikan dengan kode, lalu dirangkum dalam satu tabel di akhir bagian ini.

In [1]:
import re
import numpy as np
import pandas as pd

gdsc = pd.read_csv("GDSC_DATASET.csv")
expr_raw = pd.read_csv("CCLE_RNAseq_rsem_genes_tpm_20180929.txt.gz", sep="\t")
print("GDSC:", gdsc.shape, "| CCLE:", expr_raw.shape)

GDSC: (242035, 19) | CCLE: (57820, 1021)


## Temuan 1. Missing value pada metadata GDSC bersifat berkelompok, bukan acak

Pertanyaannya: apakah kolom metadata yang kosong menyebar acak, atau kosong bersama-sama pada *cell line* yang sama?

In [2]:
kolom_meta = ["GDSC Tissue descriptor 1", "GDSC Tissue descriptor 2", "Screen Medium",
              "Growth Properties", "CNA", "Gene Expression", "Methylation"]

print("Jumlah baris kosong per kolom metadata:")
print(gdsc[kolom_meta].isna().sum().to_string())

# apakah ketujuh kolom itu kosong di baris yang sama?
kosong_semua = gdsc[kolom_meta].isna().all(axis=1)
kosong_sebagian = gdsc[kolom_meta].isna().any(axis=1) & ~kosong_semua
print("\nBaris dengan 7 kolom kosong sekaligus:", kosong_semua.sum(), f"({kosong_semua.mean():.2%} dari seluruh baris)")
print("Baris dengan hanya sebagian kolom yang kosong:", kosong_sebagian.sum())

tanpa_meta = gdsc[kosong_semua]
print("\nJumlah cell line yang terdampak:", tanpa_meta.COSMIC_ID.nunique(), "dari", gdsc.COSMIC_ID.nunique())
print("Contoh:", tanpa_meta.CELL_LINE_NAME.drop_duplicates().head(8).tolist())

# apakah Cancer Type dan MSI ikut kosong di baris yang sama?
print("\nDi baris itu, Cancer Type dan MSI kosong semua?")
print(tanpa_meta[["Cancer Type (matching TCGA label)", "Microsatellite instability Status (MSI)"]].isna().all().to_string())

Jumlah baris kosong per kolom metadata:
GDSC Tissue descriptor 1    9366
GDSC Tissue descriptor 2    9366
Screen Medium               9366
Growth Properties           9366
CNA                         9366
Gene Expression             9366
Methylation                 9366

Baris dengan 7 kolom kosong sekaligus: 9366 (3.87% dari seluruh baris)
Baris dengan hanya sebagian kolom yang kosong: 0

Jumlah cell line yang terdampak: 39 dari 969
Contoh: ['5637', 'T24', '697', 'HuTu-80', 'SNU-1', 'SNU-5', 'SNU-16', 'NTERA-2-cl-D1']

Di baris itu, Cancer Type dan MSI kosong semua?
Cancer Type (matching TCGA label)          True
Microsatellite instability Status (MSI)    True


## Temuan 2. Duplikasi tersembunyi pada obat

Cek duplikat biasa memakai `DRUG_ID` tidak menemukan apa-apa. Di sini kami bandingkan dengan memakai `DRUG_NAME`.

In [3]:
print("Duplikat baris penuh:", gdsc.duplicated().sum())
print("Duplikat (COSMIC_ID, DRUG_ID):  ", gdsc.duplicated(["COSMIC_ID", "DRUG_ID"]).sum())
print("Duplikat (COSMIC_ID, DRUG_NAME):", gdsc.duplicated(["COSMIC_ID", "DRUG_NAME"]).sum())
print("DRUG_ID unik:", gdsc.DRUG_ID.nunique(), "| DRUG_NAME unik:", gdsc.DRUG_NAME.nunique())

# obat yang punya lebih dari satu DRUG_ID
id_per_nama = gdsc.groupby("DRUG_NAME").DRUG_ID.nunique()
obat_ganda = id_per_nama[id_per_nama > 1].index.tolist()
print("\nObat dengan lebih dari satu DRUG_ID:", obat_ganda)

# apakah hasil kedua ID sama untuk cell line yang sama?
hasil = []
for nama in obat_ganda:
    g = gdsc[gdsc.DRUG_NAME == nama]
    id1, id2 = sorted(g.DRUG_ID.unique())
    p = g.pivot_table(index="COSMIC_ID", columns="DRUG_ID", values="LN_IC50").dropna()
    hasil.append({"obat": nama, "ID_1": id1, "ID_2": id2, "sel_diuji_di_keduanya": len(p),
                  "korelasi": round(p[id1].corr(p[id2]), 2),
                  "rata2_selisih (ID_2 - ID_1)": round((p[id2] - p[id1]).mean(), 2)})
display(pd.DataFrame(hasil))

dup = gdsc[gdsc.duplicated(["COSMIC_ID", "DRUG_NAME"], keep=False)]
selisih = dup.groupby(["COSMIC_ID", "DRUG_NAME"]).LN_IC50.agg(lambda s: s.max() - s.min())
print("Selisih LN_IC50 pada pasangan ganda: median", round(selisih.median(), 2),
      "| lebih dari 1 satuan:", f"{(selisih > 1).mean():.0%}")

# spasi tersembunyi pada nama obat
spasi = gdsc.DRUG_NAME != gdsc.DRUG_NAME.str.strip()
print("\nBaris dengan spasi di ujung nama obat:", spasi.sum(), "->", gdsc.DRUG_NAME[spasi].map(repr).unique().tolist())

Duplikat baris penuh: 0
Duplikat (COSMIC_ID, DRUG_ID):   0
Duplikat (COSMIC_ID, DRUG_NAME): 6288
DRUG_ID unik: 295 | DRUG_NAME unik: 286

Obat dengan lebih dari satu DRUG_ID: ['Acetalax', 'Dactinomycin', 'Docetaxel', 'Fulvestrant', 'GSK343', 'Oxaliplatin', 'Selumetinib', 'Ulixertinib', 'Uprosertib']


,obat,ID_1,ID_2,sel_diuji_di_keduanya,korelasi,rata2_selisih (ID_2 - ID_1)
0,Acetalax,1803,1804,717,0.93,-0.38
1,Dactinomycin,1811,1911,699,0.73,-2.37
2,Docetaxel,1007,1819,669,0.83,2.22
3,Fulvestrant,1200,1816,715,0.65,1.57
4,GSK343,1627,2037,669,0.57,1.56
5,Oxaliplatin,1089,1806,717,0.88,1.08
6,Selumetinib,1062,1736,699,0.84,0.55
7,Ulixertinib,1908,2047,735,0.57,-0.55
8,Uprosertib,1553,2106,668,0.79,-0.12


Selisih LN_IC50 pada pasangan ganda: median 1.1 | lebih dari 1 satuan: 54%

Baris dengan spasi di ujung nama obat: 739 -> ["'GSK-LSD1-2HCl '"]


## Temuan 3. Ketidakkonsistenan identitas antar sumber

GDSC dan CCLE ditulis dengan format nama yang berbeda (mis. `A549` di GDSC, `A549_LUNG` di CCLE). Di sini kami cek seberapa jauh perbedaannya.

In [4]:
def norm(s):
    # huruf besar, buang semua tanda baca
    return re.sub(r"[^A-Z0-9]", "", str(s).upper())

expr = expr_raw.drop(columns="transcript_ids").set_index("gene_id")

cells = gdsc.drop_duplicates("COSMIC_ID")[["COSMIC_ID", "CELL_LINE_NAME", "GDSC Tissue descriptor 1"]].copy()
cells["key"] = cells.CELL_LINE_NAME.map(norm)

ccle = pd.DataFrame({"ccle_sample": expr.columns})
ccle["nama"] = ccle.ccle_sample.str.split("_", n=1).str[0]
ccle["jaringan"] = ccle.ccle_sample.str.split("_", n=1).str[1]
ccle["key"] = ccle.nama.map(norm)

print("Cell line GDSC:", len(cells), "| sampel CCLE:", len(ccle))
print("Nama cocok persis (tanpa normalisasi):", len(set(cells.CELL_LINE_NAME) & set(ccle.nama)))
print("Pasangan setelah nama dinormalisasi:  ", len(cells.merge(ccle, on="key")))

# dua cell line berbeda bisa jadi bernama sama setelah tanda baca dibuang
print("\nKunci ganda di GDSC:")
display(cells[cells.key.duplicated(keep=False)][["CELL_LINE_NAME", "GDSC Tissue descriptor 1", "key"]])
print("Kunci ganda di CCLE:")
display(ccle[ccle.key.duplicated(keep=False)][["ccle_sample", "key"]])

# pasangan yang sebenarnya sama tetapi tidak cocok (huruf O vs angka 0)
print("Beda huruf O dan angka 0:",
      cells.CELL_LINE_NAME[cells.CELL_LINE_NAME.isin(["786-0", "U031"])].tolist(), "vs",
      ccle.ccle_sample[ccle.nama.isin(["786O", "UO31"])].tolist())

# istilah jaringan dan ID gen
print("\nKategori jaringan: GDSC =", cells["GDSC Tissue descriptor 1"].nunique(), "| CCLE =", ccle.jaringan.nunique())
print("Contoh gene_id CCLE:", expr.index[:3].tolist(), "(kode Ensembl bernomor versi, tanpa nama gen)")

Cell line GDSC: 969 | sampel CCLE: 1019
Nama cocok persis (tanpa normalisasi): 126
Pasangan setelah nama dinormalisasi:   622

Kunci ganda di GDSC:


,CELL_LINE_NAME,GDSC Tissue descriptor 1,key
527,KM-H2,lymphoma,KMH2
642,TT,thyroid,TT
821,KMH-2,thyroid,KMH2
859,T-T,aero_dig_tract,TT


Kunci ganda di CCLE:


,ccle_sample,key
976,TT_OESOPHAGUS,TT
977,TT_THYROID,TT


Beda huruf O dan angka 0: ['786-0', 'U031'] vs ['786O_KIDNEY', 'UO31_KIDNEY']

Kategori jaringan: GDSC = 19 | CCLE = 26
Contoh gene_id CCLE: ['ENSG00000000003.10', 'ENSG00000000005.5', 'ENSG00000000419.8'] (kode Ensembl bernomor versi, tanpa nama gen)


## Temuan 4. Data berkurang dan ada bias seleksi setelah penggabungan

Temuan ini baru bisa dihitung setelah GDSC dan CCLE digabung, jadi kode buktinya ada di **Bagian 2.5** (setelah pipeline).

## Temuan 5. Ekspresi gen sangat miring, sarat nol, dan punya nilai ekstrem

In [5]:
from scipy.stats import skew

nilai = expr.to_numpy()
print("Jumlah gen:", expr.shape[0], "| jumlah sampel:", expr.shape[1])
print("NaN:", np.isnan(nilai).sum(), "| nilai negatif:", (nilai < 0).sum())

# banyak nol
print(f"\nNilai tepat 0: {(nilai == 0).mean():.1%}")
print("Gen yang nol di semua sampel:", (expr.sum(axis=1) == 0).sum())
print("Gen yang tidak pernah di atas TPM 1:", (expr.max(axis=1) <= 1).sum())

# sebaran miring dan nilai ekstrem
print("\nMedian:", np.median(nilai), "| rata-rata:", nilai.mean().round(1))
print("Persentil 99:", np.percentile(nilai, 99).round(1), "| persentil 99,9:", np.percentile(nilai, 99.9).round(1),
      "| maksimum:", nilai.max())
print("Sel dengan nilai > 100.000 TPM:", (nilai > 100000).sum())

contoh = nilai[:, :50].ravel()
print("Skewness (50 sampel pertama): data asli =", round(skew(contoh), 1),
      "| setelah log2(TPM+1) =", round(skew(np.log2(contoh + 1)), 1))

# satu gen yang mendominasi
gen_puncak = expr.idxmax()
print("\nGen tertinggi di tiap sampel (3 teratas):", gen_puncak.value_counts().head(3).to_dict())
porsi = expr.max() / expr.sum() * 100
print("Porsi total TPM milik gen teratas per sampel: median", round(porsi.median(), 1), "% | maksimum", round(porsi.max(), 1), "%")

Jumlah gen: 57820 | jumlah sampel: 1019
NaN: 0 | nilai negatif: 0



Nilai tepat 0: 54.4%


Gen yang nol di semua sampel: 8444


Gen yang tidak pernah di atas TPM 1: 18737



Median: 0.0 | rata-rata: 17.3


Persentil 99: 206.4 | persentil 99,9: 2222.4 | maksimum: 225730.0
Sel dengan nilai > 100.000 TPM: 70


Skewness (50 sampel pertama): data asli = 124.2 | setelah log2(TPM+1) = 2.1

Gen tertinggi di tiap sampel (3 teratas): {'ENSG00000228253.1': 971, 'ENSG00000198712.1': 7, 'ENSG00000156508.13': 6}


Porsi total TPM milik gen teratas per sampel: median 4.8 % | maksimum 22.6 %


## Temuan 6. Kategori tidak seimbang dan label tidak lengkap

In [6]:
cl = gdsc.drop_duplicates("COSMIC_ID")

# jaringan GDSC
jar = cl["GDSC Tissue descriptor 1"].value_counts()
print("GDSC Tissue descriptor 1 (jumlah cell line per kategori):")
print(cl["GDSC Tissue descriptor 1"].value_counts(dropna=False).to_string())
print("\nKategori terbesar:", jar.index[0], jar.iloc[0], "| terkecil:", jar.index[-1], jar.iloc[-1],
      "| rasio:", round(jar.iloc[0] / jar.iloc[-1], 1))
print("Kategori dengan kurang dari 20 cell line:", (jar < 20).sum())

# label TCGA
print("\nTCGA_DESC: UNCLASSIFIED =", (cl.TCGA_DESC == "UNCLASSIFIED").sum(), "| kosong =", cl.TCGA_DESC.isna().sum())
ct = cl["Cancer Type (matching TCGA label)"]
print("Cancer Type: kosong =", ct.isna().sum(), f"({ct.isna().mean():.1%} dari cell line)")
print("MSI:", cl["Microsatellite instability Status (MSI)"].value_counts(dropna=False).to_dict())

# jaringan CCLE (dari nama sampel)
jar_ccle = ccle.jaringan.value_counts()
print("\nJaringan CCLE: terbesar =", jar_ccle.index[0], jar_ccle.iloc[0],
      "| terkecil =", jar_ccle.index[-1], jar_ccle.iloc[-1])
print("Kategori CCLE dengan kurang dari 10 sampel:", (jar_ccle < 10).sum(), "dari", len(jar_ccle))

GDSC Tissue descriptor 1 (jumlah cell line per kategori):
GDSC Tissue descriptor 1
lung_NSCLC           107
urogenital_system    101
leukemia              81
aero_dig_tract        74
lymphoma              67
lung_SCLC             60
nervous_system        54
skin                  54
breast                52
large_intestine       46
digestive_system      42
NaN                   39
bone                  38
kidney                34
neuroblastoma         32
pancreas              29
soft_tissue           20
myeloma               17
thyroid               15
lung                   7

Kategori terbesar: lung_NSCLC 107 | terkecil: lung 7 | rasio: 15.3
Kategori dengan kurang dari 20 cell line: 3

TCGA_DESC: UNCLASSIFIED = 181 | kosong = 6
Cancer Type: kosong = 205 (21.2% dari cell line)
MSI: {'MSS/MSI-L': 858, 'MSI-H': 58, nan: 53}

Jaringan CCLE: terbesar = LUNG 188 | terkecil = SMALL_INTESTINE 1
Kategori CCLE dengan kurang dari 10 sampel: 6 dari 26


## Ringkasan Data Quality Findings

| No | Temuan | Bukti | Implikasi |
|---|---|---|---|
| 1 | Missing value pada metadata GDSC bersifat berkelompok, bukan acak | 39 dari 969 *cell line* (9.366 baris) kosong di 7 kolom metadata sekaligus (jaringan 1 dan 2, medium, sifat tumbuh, CNA, Gene Expression, Methylation). `Cancer Type` dan MSI ikut kosong di baris yang sama, dan tidak ada baris yang kosong hanya sebagian. Salah satunya `697`, sel darah menurut CCLE | Kosongnya bukan kebetulan, jadi jangan diimputasi. Bila baris tanpa jaringan dibuang, 39 sel ikut hilang. Jaringannya bisa diisi dari nama sampel CCLE (kolom `tissue_ccle`) |
| 2 | Duplikasi tersembunyi pada obat | Tidak ada duplikat jika kuncinya `DRUG_ID`, tetapi ada 6.288 pasangan ganda jika kuncinya `DRUG_NAME`: 295 `DRUG_ID` untuk 286 nama, dan sembilan obat punya dua ID (mis. Docetaxel, Oxaliplatin). Hasil kedua ID tidak sama: korelasi `LN_IC50` 0,57 sampai 0,93, selisih median 1,1, dan rata-rata bergeser hingga 2,37. Ada juga 739 baris dengan spasi di ujung nama obat (`'GSK-LSD1-2HCl '`) | Pakai `DRUG_ID` sebagai kunci, bukan nama obat. Dua ID jangan dirata-ratakan; pilih satu ID per obat atau analisis terpisah (diputuskan di Preprocessing Plan). Nama obat di-`strip()` |
| 3 | Ketidakkonsistenan identitas antar sumber | Nama *cell line* GDSC dan CCLE hanya cocok persis pada 126 sel; setelah dinormalisasi menjadi 622 pasangan. Normalisasi membuat enam pasangan ambigu (`TT` dan `T-T`; `KM-H2` dan `KMH-2`) dan melewatkan dua pasangan yang beda huruf O/angka 0 (`786-0`, `U031`). Nama jaringan juga berbeda (19 kategori di GDSC, 26 di CCLE), dan `gene_id` berupa kode Ensembl bernomor versi tanpa nama gen | Penggabungan tidak bisa memakai nama mentah. Pipeline memakai nama yang dinormalisasi, cek kecocokan jaringan untuk kunci ambigu, dan dua pemetaan manual. Untuk menafsirkan gen nanti dibutuhkan tabel Ensembl ke simbol gen |
| 4 | Data berkurang dan ada bias seleksi setelah penggabungan (bukti di Bagian 2.5) | Tersisa 621 *cell line* (64,1% dari GDSC dan 60,9% dari CCLE). Cakupan beda antar jaringan (bone 28,9%, neuroblastoma 37,5%, lung_NSCLC 86,9%); leukemia hanya 50 dari 81. Sel yang tersisa sedikit lebih resisten (median `LN_IC50` rata-rata 2,90 vs 2,63; p = 0,0003). Jumlah sel per obat turun, median dari 893 menjadi 569 | Kesimpulan hanya berlaku untuk 621 sel ini dan perlu ditulis sebagai keterbatasan. Saat memilih obat, pilih yang jumlah selnya masih banyak setelah penggabungan |
| 5 | Ekspresi gen sangat miring, sarat nol, dan punya nilai ekstrem | 54,4% nilai adalah 0 dan 8.444 gen nol di semua sampel. Median TPM 0, rata-rata 17,3, maksimum 225.730; skewness 124 pada data asli turun jadi 2,1 setelah log2. Satu gen (`ENSG00000228253.1`) menjadi yang tertinggi di 971 dari 1.019 sampel | Data tidak bisa langsung dipakai untuk PCA/UMAP. Perlu filter gen aktif, log2(TPM+1), dan standardisasi (Langkah 6 pipeline). Gen dominan perlu dicek namanya (diduga gen mitokondria, belum diverifikasi) |
| 6 | Kategori tidak seimbang dan label tidak lengkap | Jaringan GDSC punya 19 kategori, dari `lung_NSCLC` (107 sel) sampai `lung` (7 sel), rasio 15,3, dan 3 kategori di bawah 20 sel. TCGA_DESC: 181 sel `UNCLASSIFIED`. `Cancer Type` kosong di 205 sel (21,2%). MSI kosong di 53 sel, dan MSI-H hanya 58 dibanding 858 MSS/MSI-L. Di CCLE, 6 dari 26 jaringan punya kurang dari 10 sampel | Jangan pakai `Cancer Type` sebagai label utama; pakai `GDSC Tissue descriptor 1`. Kategori kecil perlu digabung atau ditafsirkan hati-hati. Label kosong diberi kategori `Unknown` |

# Bagian 2. Initial Data Pipeline

Alur pipeline:

**muat data → rapikan identitas → cocokkan GDSC dengan CCLE → bangun tabel → validasi → transformasi ekspresi**

Pipeline membaca langsung dua file dataset dan **tidak menyimpan file atau folder baru**. Hasilnya berupa DataFrame di memori notebook:

| Variabel | Isi | Ukuran |
|---|---|---|
| `ekspresi` | Matriks ekspresi TPM mentah, baris = *cell line* (indeks `COSMIC_ID`), kolom = gen | 621 × 57.820 |
| `ekspresi_std` | Matriks ekspresi setelah transformasi (Langkah 6), siap untuk reduksi dimensi | 621 × 2.000 |
| `metadata` | Satu baris per *cell line* (jaringan GDSC dan CCLE, TCGA, MSI, metode pencocokan) | 621 baris |
| `respons` | Respons obat (format panjang), hanya untuk *cell line* yang berhasil dicocokkan | 157.558 baris |
| `peta`, `jejak` | Pemetaan nama *cell line* dan catatan jumlah data di tiap langkah | |

### Parameter transformasi ekspresi

Parameter di bawah dipakai pada Langkah 6. **Angka 2.000 gen adalah asumsi kami, bukan aturan baku**, dan boleh diubah lewat `N_GEN`.

In [7]:
AMBANG_TPM = 1          # gen dianggap aktif jika TPM > 1
PROPORSI_SAMPEL = 0.10  # ... pada minimal 10% sampel
N_GEN = 2000            # ASUMSI: jumlah gen bervarians tertinggi yang dipakai (pilihan, boleh diubah)

In [8]:
log = []
def catat(langkah, **info):
    log.append({"langkah": langkah, **info})

# --- Langkah 1. Muat dan rapikan identitas -------------------------------
gdsc = pd.read_csv("GDSC_DATASET.csv")
gdsc["DRUG_NAME"] = gdsc["DRUG_NAME"].str.strip()           # 'GSK-LSD1-2HCl ' -> tanpa spasi akhir
catat("1a. GDSC dimuat", baris=len(gdsc), cell_line=gdsc.COSMIC_ID.nunique(), obat=gdsc.DRUG_ID.nunique())

expr = (pd.read_csv("CCLE_RNAseq_rsem_genes_tpm_20180929.txt.gz", sep="\t")
          .drop(columns="transcript_ids")                  # tidak dipakai
          .set_index("gene_id")
          .T.astype("float32"))                            # baris = sampel, kolom = gen
expr.index.name = "ccle_sample"
catat("1b. CCLE dimuat dan ditranspose", sampel=expr.shape[0], gen=expr.shape[1])

In [9]:
# --- Langkah 2. Kunci pencocokan ----------------------------------------
def norm(s):
    return re.sub(r"[^A-Z0-9]", "", str(s).upper())

H = "HAEMATOPOIETIC_AND_LYMPHOID_TISSUE"
# Jaringan GDSC -> jaringan CCLE yang masuk akal. Dipakai hanya untuk
# menyelesaikan kunci ambigu dan menandai pasangan yang janggal.
KOMPAT = {
    "aero_dig_tract": {"UPPER_AERODIGESTIVE_TRACT", "OESOPHAGUS", "SALIVARY_GLAND"},
    "bone": {"BONE", "SOFT_TISSUE"}, "breast": {"BREAST"},
    "digestive_system": {"STOMACH", "LIVER", "BILIARY_TRACT", "SMALL_INTESTINE", "PANCREAS", "LARGE_INTESTINE", "OESOPHAGUS"},
    "kidney": {"KIDNEY"}, "large_intestine": {"LARGE_INTESTINE"},
    "leukemia": {H}, "lymphoma": {H}, "myeloma": {H},
    "lung": {"LUNG", "PLEURA"}, "lung_NSCLC": {"LUNG", "PLEURA"}, "lung_SCLC": {"LUNG", "PLEURA"},
    "nervous_system": {"CENTRAL_NERVOUS_SYSTEM"}, "neuroblastoma": {"AUTONOMIC_GANGLIA"},
    "pancreas": {"PANCREAS"}, "skin": {"SKIN"}, "soft_tissue": {"SOFT_TISSUE", "BONE"},
    "thyroid": {"THYROID"},
    "urogenital_system": {"OVARY", "ENDOMETRIUM", "URINARY_TRACT", "PROSTATE", "CERVIX", "KIDNEY", "SOFT_TISSUE"},
}
# Pasangan sah yang terlewat normalisasi (O vs 0). Ditambahkan manual.
MANUAL = {"786-0": "786O_KIDNEY", "U031": "UO31_KIDNEY"}

cells = (gdsc.drop_duplicates("COSMIC_ID")
             [["COSMIC_ID", "CELL_LINE_NAME", "GDSC Tissue descriptor 1"]]
             .rename(columns={"GDSC Tissue descriptor 1": "tissue_gdsc"}))
cells["key"] = cells.CELL_LINE_NAME.map(norm)

ccle = pd.DataFrame({"ccle_sample": expr.index})
ccle["key"] = ccle.ccle_sample.str.split("_", n=1).str[0].map(norm)
ccle["tissue_ccle"] = ccle.ccle_sample.str.split("_", n=1).str[1]

def jaringan_cocok(t_gdsc, t_ccle):
    return t_gdsc in KOMPAT and t_ccle in KOMPAT[t_gdsc]

# --- Langkah 3. Cocokkan -------------------------------------------------
kand = cells.merge(ccle, on="key")
kand["ambigu"] = (kand.groupby("key").COSMIC_ID.transform("nunique") > 1) | \
                 (kand.groupby("key").ccle_sample.transform("nunique") > 1)
kand["tissue_cocok"] = [jaringan_cocok(a, b) for a, b in zip(kand.tissue_gdsc, kand.tissue_ccle)]
catat("3a. Kandidat setelah normalisasi nama", pasangan=len(kand), ambigu=int(kand.ambigu.sum()))

unik = kand[~kand.ambigu].assign(metode="nama_unik")
selesai = kand[kand.ambigu & kand.tissue_cocok].assign(metode="nama+jaringan")
manual = pd.DataFrame([
    {"COSMIC_ID": r.COSMIC_ID, "CELL_LINE_NAME": r.CELL_LINE_NAME, "tissue_gdsc": r.tissue_gdsc, "key": r.key,
     "ccle_sample": c, "tissue_ccle": c.split("_", 1)[1], "metode": "manual"}
    for n, c in MANUAL.items() for r in cells[cells.CELL_LINE_NAME == n].itertuples()
])
peta = pd.concat([unik, selesai, manual], ignore_index=True).drop(columns=["ambigu", "tissue_cocok"])
peta["tissue_cocok"] = [jaringan_cocok(a, b) for a, b in zip(peta.tissue_gdsc, peta.tissue_ccle)]
catat("3b. Pemetaan akhir 1:1", cell_line=len(peta), nama_unik=len(unik), nama_jaringan=len(selesai), manual=len(manual))

In [10]:
# --- Langkah 4. Bangun tiga tabel ---------------------------------------
peta = peta.sort_values("COSMIC_ID").reset_index(drop=True)

# 4a. Matriks ekspresi (baris diberi COSMIC_ID agar bisa digabung dengan GDSC)
ekspresi = expr.loc[peta.ccle_sample].copy()
ekspresi.index = peta.COSMIC_ID.values
ekspresi.index.name = "COSMIC_ID"

# 4b. Metadata satu baris per cell line
meta_cols = ["COSMIC_ID", "TCGA_DESC", "GDSC Tissue descriptor 1", "GDSC Tissue descriptor 2",
             "Cancer Type (matching TCGA label)", "Microsatellite instability Status (MSI)",
             "Screen Medium", "Growth Properties"]
metadata = (peta[["COSMIC_ID", "CELL_LINE_NAME", "ccle_sample", "tissue_ccle", "metode", "tissue_cocok"]]
            .merge(gdsc.drop_duplicates("COSMIC_ID")[meta_cols], on="COSMIC_ID", how="left"))
metadata["metadata_gdsc_lengkap"] = metadata["GDSC Tissue descriptor 1"].notna()

# 4c. Respons obat: hanya cell line yang punya ekspresi. Kunci obat = DRUG_ID.
resp_cols = ["COSMIC_ID", "DRUG_ID", "DRUG_NAME", "LN_IC50", "AUC", "Z_SCORE", "TARGET", "TARGET_PATHWAY"]
respons = gdsc.loc[gdsc.COSMIC_ID.isin(peta.COSMIC_ID), resp_cols].reset_index(drop=True)

catat("4. Tabel akhir", ekspresi_baris=ekspresi.shape[0], ekspresi_gen=ekspresi.shape[1],
      respons_baris=len(respons), respons_obat_id=respons.DRUG_ID.nunique())

In [11]:
# --- Langkah 5. Validasi (pipeline berhenti bila ada yang salah) --------
assert peta.COSMIC_ID.is_unique and peta.ccle_sample.is_unique, "pemetaan tidak 1:1"
assert ekspresi.index.is_unique, "COSMIC_ID ganda di matriks ekspresi"
assert list(ekspresi.index) == list(metadata.COSMIC_ID), "urutan ekspresi dan metadata berbeda"
assert not ekspresi.isna().any().any(), "ada NaN di matriks ekspresi"
assert (ekspresi.to_numpy() >= 0).all(), "ada TPM negatif"
assert np.allclose(ekspresi.to_numpy().sum(axis=1, dtype="float64"), 1e6, atol=10), "total TPM per sampel menyimpang"  # float64: penjumlahan float32 menumpuk galat
assert set(respons.COSMIC_ID) <= set(ekspresi.index), "respons memuat sel tanpa ekspresi"
assert not respons.duplicated(["COSMIC_ID", "DRUG_ID"]).any(), "pasangan sel-obat ganda"
print("Semua validasi lulus.")

print("\nSel dengan jaringan GDSC tidak cocok CCLE atau kosong (perlu diperiksa):",
      (~metadata.tissue_cocok).sum())
display(metadata.loc[~metadata.tissue_cocok & metadata.metadata_gdsc_lengkap,
                     ["CELL_LINE_NAME", "GDSC Tissue descriptor 1", "tissue_ccle"]])
print("Sel tanpa metadata GDSC tetapi punya ekspresi (jaringan bisa diambil dari CCLE):",
      (~metadata.metadata_gdsc_lengkap).sum())

Semua validasi lulus.

Sel dengan jaringan GDSC tidak cocok CCLE atau kosong (perlu diperiksa): 19


,CELL_LINE_NAME,GDSC Tissue descriptor 1,tissue_ccle
242,G-401,kidney,SOFT_TISSUE


Sel tanpa metadata GDSC tetapi punya ekspresi (jaringan bisa diambil dari CCLE): 18


### Langkah 6. Transformasi ekspresi

Urutannya:

1. **Filter gen aktif**: simpan gen dengan TPM > 1 pada minimal 10% sampel. Gen yang hampir selalu 0 tidak membawa informasi dan membuat matriks sangat jarang.
2. **log2(TPM + 1)**: meredam nilai ekstrem dan sebaran yang sangat miring (lihat Temuan 5). Angka +1 supaya nilai 0 tidak menjadi tak hingga.
3. **Pilih 2.000 gen bervarians tertinggi** (`N_GEN`, asumsi): gen dengan variasi terbesar antar sampel paling berguna untuk membedakan kelompok. Varians dihitung pada data log2 **sebelum** standardisasi, karena sesudah standardisasi semua gen bervarians 1 sehingga tidak bisa dibedakan lagi.
4. **Standardisasi per gen**: setiap gen diubah menjadi rata-rata 0 dan simpangan baku 1, supaya gen dengan nilai besar tidak mendominasi jarak antar sampel.

In [12]:
# --- Langkah 6. Transformasi ekspresi -------------------------------------
print("Gen yang nol di semua 621 sampel:", int((ekspresi.sum(axis=0) == 0).sum()))

# 6a. Filter gen aktif (TPM > 1 pada minimal 10% sampel)
persen_aktif = (ekspresi > AMBANG_TPM).mean(axis=0)
gen_aktif = persen_aktif[persen_aktif >= PROPORSI_SAMPEL].index
ekspresi_aktif = ekspresi[gen_aktif]
catat("6a. Filter gen aktif", gen_sebelum=ekspresi.shape[1], gen_sesudah=ekspresi_aktif.shape[1])

# 6b. log2(TPM + 1)
ekspresi_log = np.log2(ekspresi_aktif + 1)

# 6c. Pilih N_GEN gen bervarians tertinggi (varians dihitung pada data log2, sebelum standardisasi)
assert N_GEN <= ekspresi_log.shape[1], "N_GEN lebih besar dari jumlah gen aktif"
varians = ekspresi_log.var(axis=0)
gen_terpilih = varians.nlargest(N_GEN).index
ekspresi_top = ekspresi_log[gen_terpilih]
catat("6c. Pilih gen bervarians tertinggi", gen=ekspresi_top.shape[1])

# 6d. Standardisasi per gen (rata-rata 0, simpangan baku 1)
ekspresi_std = (ekspresi_top - ekspresi_top.mean(axis=0)) / ekspresi_top.std(axis=0, ddof=0)
catat("6d. Standardisasi", sampel=ekspresi_std.shape[0], gen=ekspresi_std.shape[1])

# validasi hasil transformasi
assert ekspresi_std.shape == (len(ekspresi), N_GEN)
assert not ekspresi_std.isna().any().any(), "ada NaN setelah standardisasi"
assert np.allclose(ekspresi_std.mean(axis=0), 0, atol=1e-4), "rata-rata per gen tidak 0"
assert np.allclose(ekspresi_std.std(axis=0, ddof=0), 1, atol=1e-3), "simpangan baku per gen tidak 1"
print("Validasi transformasi lulus.")
print(f"Gen: {ekspresi.shape[1]:,} -> aktif {ekspresi_aktif.shape[1]:,} -> dipilih {ekspresi_std.shape[1]:,}")
print("Bentuk ekspresi_std:", ekspresi_std.shape)
print("Varians gen terpilih (log2): terendah", round(varians[gen_terpilih].min(), 2), "| tertinggi", round(varians[gen_terpilih].max(), 2))

Gen yang nol di semua 621 sampel: 8904


Validasi transformasi lulus.
Gen: 57,820 -> aktif 19,879 -> dipilih 2,000
Bentuk ekspresi_std: (621, 2000)
Varians gen terpilih (log2): terendah 2.7 | tertinggi 15.97


### Bagian 2.5. Bukti Temuan 4: data berkurang dan bias seleksi

In [13]:
from scipy.stats import mannwhitneyu

semua_cl = gdsc.drop_duplicates("COSMIC_ID")
print(f"Cell line tersisa: {len(peta)} = {len(peta) / len(semua_cl):.1%} dari GDSC ({len(semua_cl)}) "
      f"dan {len(peta) / len(ccle):.1%} dari CCLE ({len(ccle)})")

# cakupan per jaringan
total = semua_cl["GDSC Tissue descriptor 1"].fillna("Unknown").value_counts()
tersisa = metadata["GDSC Tissue descriptor 1"].fillna("Unknown").value_counts()
cov = pd.DataFrame({"total_GDSC": total, "tersisa": tersisa}).fillna(0).astype(int)
cov["persen"] = (cov.tersisa / cov.total_GDSC * 100).round(1)
print("\nCakupan terendah per jaringan:")
display(cov.sort_values("persen").head(5))
print("Sel darah:")
display(cov.loc[["leukemia", "lymphoma", "myeloma"]])

# apakah sel yang tersisa mewakili GDSC? (bandingkan respons obat)
rata_sel = gdsc.groupby("COSMIC_ID").LN_IC50.mean()
tercakup = rata_sel.index.isin(peta.COSMIC_ID)
p = mannwhitneyu(rata_sel[tercakup], rata_sel[~tercakup]).pvalue
print("\nMedian LN_IC50 rata-rata per sel: tersisa =", round(rata_sel[tercakup].median(), 2),
      "| tidak tersisa =", round(rata_sel[~tercakup].median(), 2), "| p (Mann-Whitney) =", round(p, 4))

# jumlah sel per obat sebelum dan sesudah digabung
awal = gdsc.groupby("DRUG_ID").COSMIC_ID.nunique()
akhir = respons.groupby("DRUG_ID").COSMIC_ID.nunique()
print("Sel per obat sebelum: minimum", awal.min(), "| median", int(awal.median()))
print("Sel per obat sesudah: minimum", akhir.min(), "| median", int(akhir.median()))

Cell line tersisa: 621 = 64.1% dari GDSC (969) dan 60.9% dari CCLE (1019)

Cakupan terendah per jaringan:


,total_GDSC,tersisa,persen
GDSC Tissue descriptor 1,,,
bone,38,11,28.9
neuroblastoma,32,12,37.5
Unknown,39,18,46.2
kidney,34,16,47.1
aero_dig_tract,74,39,52.7


Sel darah:


,total_GDSC,tersisa,persen
GDSC Tissue descriptor 1,,,
leukemia,81,50,61.7
lymphoma,67,37,55.2
myeloma,17,14,82.4



Median LN_IC50 rata-rata per sel: tersisa = 2.9 | tidak tersisa = 2.63 | p (Mann-Whitney) = 0.0003
Sel per obat sebelum: minimum 225 | median 893
Sel per obat sesudah: minimum 115 | median 569


In [14]:
# Rekap jumlah data per langkah
jejak = pd.DataFrame({
    "langkah": [d["langkah"] for d in log],
    "keterangan": [", ".join(f"{k} = {v}" for k, v in d.items() if k != "langkah") for d in log],
})
pd.set_option("display.max_colwidth", None)
display(jejak)

,langkah,keterangan
0,1a. GDSC dimuat,"baris = 242035, cell_line = 969, obat = 295"
1,1b. CCLE dimuat dan ditranspose,"sampel = 1019, gen = 57820"
2,3a. Kandidat setelah normalisasi nama,"pasangan = 622, ambigu = 6"
3,3b. Pemetaan akhir 1:1,"cell_line = 621, nama_unik = 616, nama_jaringan = 3, manual = 2"
4,4. Tabel akhir,"ekspresi_baris = 621, ekspresi_gen = 57820, respons_baris = 157558, respons_obat_id = 295"
5,6a. Filter gen aktif,"gen_sebelum = 57820, gen_sesudah = 19879"
6,6c. Pilih gen bervarians tertinggi,gen = 2000
7,6d. Standardisasi,"sampel = 621, gen = 2000"


**Catatan untuk anggota kelompok**

- `ekspresi_std` (621 × 2.000) adalah hasil akhir pipeline dan bisa langsung dipakai untuk PCA/UMAP. `ekspresi` (TPM mentah) tetap disimpan bila ada langkah yang perlu dikerjakan ulang dengan pengaturan lain.
- Jumlah gen (`N_GEN`) bisa diubah di sel parameter. Setelah itu jalankan ulang Langkah 6.
- Kunci obat adalah `DRUG_ID`. Keputusan untuk sembilan obat dengan dua ID (pilih satu atau analisis terpisah) ada di Preprocessing Plan.
- Untuk 18 sel yang tidak punya metadata GDSC tetapi punya ekspresi, kolom `tissue_ccle` bisa dipakai sebagai label jaringan cadangan.
- Jalankan sel pipeline dari atas ke bawah agar semua anggota memakai 621 sel yang sama.